# Module 1: The Agent Loop & Built-in Tools

Anthropic Agent SDK Enterprise Training — Module 1 of 5

The Agent SDK handles the iterative tool-use loop automatically, allowing Claude to reason, act, observe, and adjust without manual intervention.

## Concepts

The difference between the standard Client SDK and the Agent SDK; the `query()` function, agent runtime, and `allowed_tools` configuration.

## Lab 1: The Local Explorer

### Task

Initialize an Agent SDK project and configure an agent with access only to the `Read`, `Glob`, and `Grep` built-in tools.

### Execution

Prompt the agent to scan a local codebase, locate TODO or FIXME comments, and generate a Markdown summary.

### Outcome

A single-agent loop that safely navigates local file systems without custom tool implementation.


# 1. Learning Objectives & Prerequisites

---

### Course Metadata

* **Estimated Completion Time:** 25 – 30 minutes
* **Target Audience:** Software Engineers, AI Engineers, Platform Engineers, Developer Advocates, Technical Consultants
* **Prerequisites:** Python 3.10+, Git, REST APIs, `asyncio` & async generator basics
* **Required Software:** Python 3.10+, `claude-agent-sdk`, `rich`, `python-dotenv`, `uv` (recommended)
* **Expected Knowledge:** Core Python, async programming, basic CLI usage

---

### Key Learning Outcomes

By the end of this lab you will be able to:

* **Explain the Agent Loop:** Understand how Claude evaluates context, emits tool execution requests, ingests tool outputs, and adjusts plans iteratively.
* **Explain the difference between Client SDK and Agent SDK:** Articulate when to use raw message creation API calls versus automated agent runtime orchestration.
* **Configure `ClaudeAgentOptions`:** Define system-level prompt guidelines, target model selection, and tool execution security boundaries.
* **Use `query()`:** Invoke the primary asynchronous generator interface to execute agentic tasks and process stream events.
* **Configure `allowed_tools`:** Implement defense-in-depth permission boundaries restricting agent execution to specific built-in tools (`Read`, `Glob`, `Grep`).
* **Inspect `ResultMessage`:** Handle discriminated SDK event messages (`SystemMessage`, `AssistantMessage`, `ResultMessage`) and evaluate status codes.
* **Build an autonomous local agent:** Construct a production-ready agent capable of scanning codebases, analyzing code issues, and generating structured markdown reports.


# 2. Concept Introduction: What is the Anthropic Agent SDK?

### What is the Agent SDK?

The **Anthropic Agent SDK** (`claude_agent_sdk`) acts as an event loop execution environment for tool use—repeatedly allowing Claude to reason, execute local actions, observe tool results, and adjust its plan until a task is completed.

Rather than manually calling the API, parsing `tool_use` blocks, executing functions, and appending `tool_result` messages back into conversation history, the SDK orchestrates the entire cycle within a single, high-level execution context.

### What Problem Does It Solve?

When building multi-turn AI agents with standard API clients, developers must write extensive boilerplate code to:
1. Parse `tool_use` JSON blocks returned by the model.
2. Map tool names to local Python functions and execute them safely.
3. Format return values into `tool_result` blocks.
4. Append assistant messages and tool results back into the conversation history array.
5. Manage retry loops, error handling, token budgeting, and turn limits.
6. Prevent infinite loops and state corruption.

The Agent SDK abstracts this operational complexity into a single, high-level asynchronous generator call: `query()`.

### Why Was It Built?

Anthropic developed the Agent SDK to enable developers to build agentic coding assistants and complex autonomous workflows with enterprise-grade safety, security boundaries, and minimal boilerplate.

### Client SDK vs. Agent SDK

| Dimension | Standard Client SDK (`anthropic`) | Agent SDK (`claude_agent_sdk`) |
|-----------|----------------------------------|--------------------------------|
| **Loop Control** | Manual — developer writes the request-response loop | Automatic — `query()` orchestrates multi-turn cycles internally |
| **State Management** | Manual — caller appends assistant messages and tool results | Automatic — SDK maintains context history and auto-compacts |
| **Tool Execution** | External — caller receives schema request and executes code | Local — SDK dispatches built-in and custom tools locally |
| **Primary Use Case** | Single-turn generations, strict custom pipelines | Autonomous multi-step workflows, agentic coding tasks |

When using the **Client SDK**, tool execution requires explicit loop management. When Claude returns a `tool_use` block, your code must parse the request, call the local function, format a `tool_result` message, append both messages to the request array, and invoke `messages.create()` again. This grants granular control but requires significant boilerplate.

When using the **Agent SDK**, calling `query()` delegates loop orchestration to the framework. You configure permitted capabilities via `ClaudeAgentOptions`, and the engine handles tool execution, context formatting, and iteration limits automatically.

---

### High-Level Agent Loop Architecture

```mermaid
flowchart LR
    A["Your prompt"] --> B

    subgraph agentic_loop ["agentic loop"]
        B["Claude evaluates"] -->|"tool calls"| C["Tool call(s)"]
        C -->|"tool result"| B
    end

    B -->|"no tool calls"| D["Final answer"]

    style A fill:#ececec,stroke:#ccc,color:#333
    style B fill:#d8e8d8,stroke:#b5d0b5,color:#222
    style C fill:#d8e8d8,stroke:#b5d0b5,color:#222
    style D fill:#ececec,stroke:#ccc,color:#333
    style agentic_loop fill:none,stroke:#ccc,stroke-dasharray: 5 5,color:#888
```

#### Diagram Explanation:
1. **Input (`Your prompt`):** The application invokes `query()` with a task prompt and configuration options.
2. **Evaluation (`Claude evaluates`):** Claude reviews context and determines if tools are required.
3. **Execution & Observation (`Tool call(s)` $\leftrightarrow$ `tool result`):** If tool calls are generated, the SDK intercepts them, verifies permission against `allowed_tools`, executes the tool locally, feeds back the `tool_result`, and triggers the next reasoning step.
4. **Termination (`Final answer`):** Once no further tool calls are required, Claude returns a text response and the SDK yields `ResultMessage` containing final output.


# 3. Architecture Walkthrough & Workflows

To master the Agent SDK, let us analyze the complete execution pipeline and internal turn lifecycle.

---

### Workflow 1: Overall Processing Pipeline

```mermaid
flowchart TD
    A(["User provides task"]) --> B["Configure ClaudeAgentOptions<br/>(system prompt + allowed tools)"]
    B --> C["Pass task to query()"]
    C --> D{"Claude evaluates state:<br/>select tool call"}
    D -->|"Read file"| E["Read tool returns<br/>file contents"]
    D -->|"Glob pattern"| F["Glob tool returns<br/>matching paths"]
    D -->|"Grep regex"| G["Grep tool returns<br/>matching lines"]
    E --> H["SDK appends tool_result<br/>to context"]
    F --> H
    G --> H
    H --> I{"Task complete?"}
    I -->|"No — iterate turn"| D
    I -->|"Yes — stop condition"| J(["Return final answer<br/>via ResultMessage"])

    style A fill:#e3f2fd,stroke:#1565c0,color:#0d47a1
    style B fill:#f5f5f5,stroke:#616161,color:#212121
    style C fill:#fff3e0,stroke:#e65100,color:#bf360c
    style D fill:#fce4ec,stroke:#c62828,color:#b71c1c
    style E fill:#f5f5f5,stroke:#616161,color:#212121
    style F fill:#f5f5f5,stroke:#616161,color:#212121
    style G fill:#f5f5f5,stroke:#616161,color:#212121
    style H fill:#fff3e0,stroke:#e65100,color:#bf360c
    style I fill:#fce4ec,stroke:#c62828,color:#b71c1c
    style J fill:#e8f5e9,stroke:#2e7d32,color:#1b5e20
```

#### Detailed Stage Analysis:
* **Stage A → B:** Application defines a natural-language prompt and instantiates `ClaudeAgentOptions` with a system prompt and allowed-tools whitelist (`["Read", "Glob", "Grep"]`).
* **Stage B → C:** `query()` is invoked, initializing the SDK session.
* **Stage C → D:** Claude evaluates prompt context and selects appropriate search/read tool actions.
* **Stage D → E/F/G:** Local execution of specific built-in tools:
  * `Read`: Reads content from target files.
  * `Glob`: Matches file paths using wildcard glob patterns.
  * `Grep`: Performs regex searches across directory files.
* **Stage E/F/G → H:** Results are captured, serialized as `tool_result` blocks, and appended to context history.
* **Stage H → I:** Claude checks whether user goals are fulfilled. If incomplete, another turn begins; if complete, it prepares a final answer.
* **Stage I → J:** Final output text is generated and delivered via `ResultMessage`.

---

### Workflow 2: The Agent Loop Internals

```mermaid
flowchart TD
    A["query(prompt, options)"] --> B["SDK sends prompt & tool definitions to Claude"]
    B --> C{"Claude response contains<br/>tool_use block?"}
    C -->|"Yes"| D["SDK executes requested<br/>tool locally"]
    D --> E["SDK appends tool_result<br/>to message history"]
    E --> B
    C -->|"No — text response"| F["SDK yields ResultMessage<br/>with final output"]

    style A fill:#e3f2fd,stroke:#1565c0,color:#0d47a1
    style B fill:#f5f5f5,stroke:#616161,color:#212121
    style C fill:#fce4ec,stroke:#c62828,color:#b71c1c
    style D fill:#fff3e0,stroke:#e65100,color:#bf360c
    style E fill:#f5f5f5,stroke:#616161,color:#212121
    style F fill:#e8f5e9,stroke:#2e7d32,color:#1b5e20
```

The execution loop operates across **five distinct phases**:

1. **Initialization (`query`)** — You invoke `query()` passing the user prompt and `ClaudeAgentOptions`. The SDK yields a `SystemMessage` with subtype `"init"` containing session metadata.
2. **Prompt Evaluation** — The SDK formats conversation history, system instructions, and tool definitions, sending them to Claude. Claude evaluates the context to determine the next action.
3. **Tool Execution (`tool_use`)** — If Claude requests tool execution via a `tool_use` block, the SDK validates permissions against `allowed_tools` and executes the function on your local filesystem.
4. **Observation (`tool_result`)** — The SDK serializes the tool result into a `tool_result` block, appends it to conversation history, and immediately triggers the next turn.
5. **Termination & Output** — Steps 2–4 repeat until Claude produces a response without tool calls or reaches a turn/budget limit. The SDK yields an `AssistantMessage` followed by a final `ResultMessage`.

---

### Educational Callouts

> [!NOTE]
> ### Why this matters
> Writing manual orchestrators for multi-turn tool interactions requires extensive boilerplate for state management, error handling, and message serialization. The Agent SDK abstracts these operational details into an automated runtime while keeping permission boundaries and budget controls fully explicit.

> [!NOTE]
> ### Under the hood
> Each cycle of tool invocation and observation constitutes one **turn**. When multiple read-only tools are requested in a single turn (such as reading two independent files), the SDK executes them concurrently. State-modifying operations are executed sequentially to prevent race conditions.

> [!NOTE]
> ### Common misconception
> The Agent SDK does not replace the Client SDK; it operates at a higher level of abstraction. Use the Client SDK when you require custom message-level control or single-turn API access. Use the Agent SDK when building autonomous agents that need environment interaction, tool execution, and self-directed multi-step task completion.

> [!NOTE]
> ### Design rationale
> Restricting available capabilities via `allowed_tools` provides defense-in-depth. Tools omitted from `allowed_tools` cannot be executed automatically. This ensures an exploratory agent restricted to `Read`, `Glob`, and `Grep` cannot modify files or execute arbitrary shell commands.


# 4. Problem Statement: Lab 1 — The Local Explorer

### Scenario Overview

Consider building an automated code audit agent. The agent must explore an unknown codebase, locate targeted patterns (such as `TODO` and `FIXME` comments), inspect surrounding code context, and synthesize its findings into a structured report—without hardcoding file paths or manually managing conversation state.

### Specification Matrix

| Component | Description |
|-----------|-------------|
| **Inputs** | 1. System prompt defining agent constraints.<br/>2. User task prompt describing audit goals.<br/>3. Target filesystem directory path (`data/`).<br/>4. Permitted tools list (`["Read", "Glob", "Grep"]`).<br/>5. `ANTHROPIC_API_KEY` for API request authorization. |
| **Outputs** | Structured Markdown document saved to `todo_fixme_report.md` detailing relative file paths, line numbers, comment text, and summaries. |
| **Workflow Stages** | **Stage 1:** Agent configuration (`ClaudeAgentOptions`).<br/>**Stage 2:** Autonomous loop execution (`query()`).<br/>**Stage 3:** Structured report synthesis and file persistence. |
| **Expected Behavior** | Autonomous directory traversal, regex pattern search, targeted file reading, and structured markdown compilation. |
| **Constraints** | Strictly read-only file access (`Read`, `Glob`, `Grep`). No file writing, editing, or bash execution permitted during search phase. |
| **Success Criteria** | Autonomous single-agent execution returning a complete markdown summary report without runtime exceptions or permission violations. |


# 5. Environment & Dependencies Setup

Before executing agent loops, we setup our environment, install required packages, create local fixture files, import SDK components, and verify API authentication.

---

### Step 5.1: Install Dependencies

We install:
* `claude-agent-sdk`: Core Anthropic Agent SDK framework package.
* `rich`: Terminal text formatting and markdown rendering library.
* `python-dotenv`: Environment variable loader from `.env` files.


In [ ]:
# Install dependencies using uv (recommended for speed) or pip
!uv pip install -q claude-agent-sdk rich python-dotenv || pip install -q claude-agent-sdk rich python-dotenv


: 

### Expected Output

```
[notice] A new release of pip is available: ...
Successfully installed claude-agent-sdk rich python-dotenv
```

* **What learners should see:** Clean installation output without error messages.
* **What indicates success:** Packages install successfully into the active environment.
* **Common failure modes:** Network timeout or permission denied error.
* **How to debug:** Run `pip install claude-agent-sdk rich python-dotenv --user` if running into permission boundaries.


### Step 5.2: Create Local Codebase Fixtures (`data/`)

To ensure this notebook is 100% reproducible and standalone without external repository dependencies, we generate sample codebase files inside a `data/` directory containing intentional `TODO` and `FIXME` comments.


In [ ]:
import os

# Create nested data directory structure
os.makedirs("data/auth", exist_ok=True)
os.makedirs("data/utils", exist_ok=True)

# Sample 1: data/app.py
with open("data/app.py", "w", encoding="utf-8") as f:
    f.write('''"""
Main Web Server Module
"""

def start_server():
    # TODO: Add SSL certificate configuration for HTTPS support
    print("Starting web server on port 8080...")

def handle_request(request):
    # FIXME: Sanitize incoming request headers to prevent header injection vulnerabilities
    return {"status": 200, "body": "OK"}
''')

# Sample 2: data/auth/jwt.py
with open("data/auth/jwt.py", "w", encoding="utf-8") as f:
    f.write('''"""
JWT Token Processing Module
"""

def generate_token(user_id):
    # TODO: Implement token expiration timeout (currently tokens never expire)
    return f"mock-jwt-token-for-{user_id}"

def verify_token(token):
    # FIXME: Replace mock signature verification with actual HMAC-SHA256 validation
    return True
''')

# Sample 3: data/utils/db.py
with open("data/utils/db.py", "w", encoding="utf-8") as f:
    f.write('''"""
Database Connection Utility
"""

def connect_db():
    # TODO: Implement connection pooling mechanism to optimize database throughput
    print("Connecting to database...")
''')

print("Sample codebase fixtures created successfully in directory: 'data/'")


### Expected Output

```
Sample codebase fixtures created successfully in directory: 'data/'
```

* **What learners should see:** Confirmation message indicating fixture file creation.
* **What indicates success:** Files `data/app.py`, `data/auth/jwt.py`, and `data/utils/db.py` exist on disk.
* **Common failure modes:** Permission denied error writing to local disk.
* **How to debug:** Verify write permissions in current workspace directory.


### Step 5.3: Import Libraries & SDK Primitives

Import standard Python libraries along with `claude_agent_sdk` primitives:
* `query`: Asynchronous generator entrypoint.
* `ClaudeAgentOptions`: Execution options and tool permission whitelist configuration.
* `AssistantMessage`, `ResultMessage`: Stream message event classes.


In [ ]:
import os
import asyncio
from dotenv import load_dotenv
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, ResultMessage
from rich.console import Console
from rich.markdown import Markdown

print("SDK and utility imports successful.")


### Expected Output

```
SDK and utility imports successful.
```

* **What learners should see:** Clean import execution without warnings.
* **What indicates success:** No `ModuleNotFoundError` or `ImportError`.
* **Common failure modes:** `ModuleNotFoundError: No module named 'claude_agent_sdk'`.
* **How to debug:** Re-run package installation cell (Step 5.1) and restart notebook kernel.


### Step 5.4: Load and Verify Anthropic API Key

Load environment variables from `.env` and verify authorization key presence.


In [ ]:
load_dotenv()

api_key = os.getenv("ANTHROPIC_API_KEY")
if not api_key:
    print("WARNING: ANTHROPIC_API_KEY environment variable is missing.")
    print("Please set ANTHROPIC_API_KEY in your environment or .env file before making query() calls.")
else:
    masked = api_key[:7] + "..." + api_key[-4:] if len(api_key) > 11 else "***"
    print(f"Anthropic API key successfully loaded: {masked}")


### Expected Output

```
Anthropic API key successfully loaded: sk-ant-...1234
```

* **What learners should see:** Masked API key confirmation message.
* **What indicates success:** Key present and verified.
* **Common failure modes:** `WARNING: ANTHROPIC_API_KEY environment variable is missing.`
* **How to debug:** Ensure `.env` file exists in notebook root directory with `ANTHROPIC_API_KEY=sk-ant-api03-...`.


# 6. Guided Implementation: Step-by-Step

---

### Step 1 — Initialize Agent Options (`ClaudeAgentOptions`)

#### Learning Goal
Learn how to configure `ClaudeAgentOptions` to define model execution parameters and restrict tool access to read-only utilities.

#### Explanation
`ClaudeAgentOptions` configures agent system parameters. Setting `allowed_tools=["Read", "Glob", "Grep"]` enforces strict permission boundaries—permitting search and reading while preventing file modification or command execution.


In [ ]:
TARGET_DIR = "data"  # Path to target codebase directory

# Define execution options and whitelist read-only tools
options = ClaudeAgentOptions(
    allowed_tools=["Read", "Glob", "Grep"],
    model="claude-sonnet-4-5",  # Pin a specific model; omit to use SDK default
)

console = Console()
console.print("[bold green]Agent options initialized.[/bold green]")
console.print(f"Target Directory: [cyan]{TARGET_DIR}[/cyan]")
console.print(f"Allowed tools whitelist: [yellow]{options.allowed_tools}[/yellow]")


### Expected Output

```
Agent options initialized.
Target Directory: data
Allowed tools whitelist: ['Read', 'Glob', 'Grep']
```

#### Why this works
Specifying `allowed_tools` configures the SDK runtime interceptor to automatically deny any tool execution request outside the whitelist.

#### Common mistakes
- Omitting `allowed_tools` or passing an empty array `allowed_tools=[]` — prevents Claude from calling required search tools.

#### Reflection
*Why is explicit tool permissioning safer than granting an agent unrestricted access to all local tools?*


### Step 2 — Define the Natural-Language Task Prompt

#### Learning Goal
Frame clear natural-language prompts that specify objectives, constraints, and output formats without hardcoding execution steps.

#### Explanation
High-quality prompts for autonomous agents focus on **objectives and output specifications**, leaving step-by-step tool selection strategy to the agent loop.


In [ ]:
TASK = f"""
Scan the codebase located at '{TARGET_DIR}' and identify all TODO and FIXME comments.

For each match, detail:
- Relative file path
- Line number
- Comment text
- Brief summary of the task or issue described

Organize output into a clean markdown document grouped by file.
"""

print("Task prompt defined:")
print("-" * 50)
print(TASK.strip())
print("-" * 50)


### Expected Output

```
Task prompt defined:
--------------------------------------------------
Scan the codebase located at 'data' and identify all TODO and FIXME comments.

For each match, detail:
- Relative file path
- Line number
- Comment text
- Brief summary of the task or issue described

Organize output into a clean markdown document grouped by file.
--------------------------------------------------
```

#### Why this works
The prompt defines clear requirements (target directory, required metadata fields, markdown output format) while allowing Claude to decide whether to run `Glob`, `Grep`, or `Read` first.

#### Common mistakes
- Hardcoding specific file paths in prompt, which limits the agent's exploratory autonomy.

#### Reflection
*How does goal-oriented task prompting differ from procedural step-by-step tool scripts?*


### 💡 Interactive Knowledge Check 1

> [!TIP]
> **Question 1:** Which tool is Claude most likely to call first when given the task prompt above?
>
> *Options:*
> A. `Read` — to inspect files one by one sequentially.  
> B. `Glob` or `Grep` — to discover file paths or search pattern matches across the directory quickly.  
> C. `Edit` — to modify the comments immediately.  
>
> <details>
> <summary><b>Click to reveal Answer & Explanation</b></summary>
> <b>Correct Answer: B</b><br/>
> Claude typically initiates codebase exploration using <code>Glob</code> (to find file paths) or <code>Grep</code> (to locate regex pattern matches across multiple files), before using <code>Read</code> to examine specific file contexts.
> </details>


### Step 3 — Run the Agent Loop (`query()`)

#### Learning Goal
Execute `query()` inside an asynchronous function, stream progress events (`AssistantMessage`), and extract final results from `ResultMessage`.

#### Loop Execution Diagram

```mermaid
flowchart LR
    A["query(prompt, options)"] --> B["Claude: request Glob"]
    B --> C["SDK: execute Glob"]
    C --> D["Claude: request Grep"]
    D --> E["SDK: execute Grep"]
    E --> F["Claude: request Read"]
    F --> G["SDK: execute Read"]
    G --> H["Claude: final output"]
    H --> I["Yield ResultMessage"]

    style A fill:#e3f2fd,stroke:#1565c0,color:#0d47a1
    style B fill:#fff3e0,stroke:#e65100,color:#bf360c
    style C fill:#f5f5f5,stroke:#616161,color:#212121
    style D fill:#fff3e0,stroke:#e65100,color:#bf360c
    style E fill:#f5f5f5,stroke:#616161,color:#212121
    style F fill:#fff3e0,stroke:#e65100,color:#bf360c
    style G fill:#f5f5f5,stroke:#616161,color:#212121
    style H fill:#e8f5e9,stroke:#2e7d32,color:#1b5e20
    style I fill:#e8f5e9,stroke:#2e7d32,color:#1b5e20
```

#### Explanation
`query()` yields events as the loop executes:
- `AssistantMessage`: Contains tool execution requests (`block.type == "tool_use"`).
- `ResultMessage`: Emitted upon loop termination. Check `message.subtype == "success"` before reading `message.result`.


In [ ]:
async def execute_audit(task_prompt: str, agent_options: ClaudeAgentOptions) -> str:
    final_output = ""
    console.print("[bold blue]Starting agent loop via query()...[/bold blue]")

    async for message in query(prompt=task_prompt, options=agent_options):
        # Live progress: print what Claude is doing each turn
        if isinstance(message, AssistantMessage):
            tool_calls = [
                block.name
                for block in message.content
                if hasattr(block, "type") and block.type == "tool_use"
            ]
            if tool_calls:
                console.print(f"[dim]  → Tool calls requested: {', '.join(tool_calls)}[/dim]")

        # Final result: check subtype before accessing .result
        if isinstance(message, ResultMessage):
            if message.subtype == "success":
                # .result is only present on the success variant
                final_output = message.result or ""
                console.print("[bold green]✓ Execution completed successfully.[/bold green]")
            else:
                final_output = f"Execution stopped with status: {message.subtype}"
                console.print(f"[bold red]✗ Execution stopped: {message.subtype}[/bold red]")

    return final_output


def _run_execute_audit() -> str:
    if os.name == "nt":
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    return asyncio.run(execute_audit(TASK, options))


# Execute the asynchronous query loop in a worker thread with subprocess support
try:
    # In Jupyter, await the async function directly (avoid asyncio.run in a worker thread)
    response_text = await execute_audit(TASK, options)
except Exception as e:
    response_text = f"Execution failed: {e}"
    console.print(f"[bold red]✗ Execution failed: {e}[/bold red]")

console.print("\n[bold cyan]--- Agent Response ---[/bold cyan]\n")
console.print(Markdown(response_text))

### Expected Output

```
Starting agent loop via query()...
  → Tool calls requested: Glob
  → Tool calls requested: Grep
  → Tool calls requested: Read
✓ Execution completed successfully.

--- Agent Response ---

# Codebase TODO / FIXME Audit

## data/app.py
- **Line 6:** `# TODO: Add SSL certificate configuration for HTTPS support`
  - *Summary:* Needs SSL configuration for secure HTTPS connections.
- **Line 10:** `# FIXME: Sanitize incoming request headers to prevent header injection vulnerabilities`
  - *Summary:* Header injection vulnerability risk; requires sanitization.

## data/auth/jwt.py
- **Line 6:** `# TODO: Implement token expiration timeout (currently tokens never expire)`
  - *Summary:* Tokens lack expiration timeout.
- **Line 10:** `# FIXME: Replace mock signature verification with actual HMAC-SHA256 validation`
  - *Summary:* Insecure JWT signature verification mock.

## data/utils/db.py
- **Line 6:** `# TODO: Implement connection pooling mechanism to optimize database throughput`
  - *Summary:* Connection pooling optimization required.
```

#### Why this works
`query()` encapsulates the entire loop. `AssistantMessage` provides real-time tool tracking, and `ResultMessage` delivers final structured report text safely.

#### Common mistakes
- Trying to access `message.result` directly without checking `isinstance(message, ResultMessage)` or checking `message.subtype == "success"`.

#### Reflection
*What advantages does event-driven stream iteration offer over standard single-call blocking APIs?*


### Step 4 — Save the Generated Report

#### Learning Goal
Persist agent audit results to a local Markdown file.


In [ ]:
REPORT_PATH = "todo_fixme_report.md"

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write("# Codebase TODO / FIXME Audit Report\n\n")
    f.write(f"Target directory: `{TARGET_DIR}`\n\n")
    f.write(response_text)

console.print(f"[bold green]Audit report written to {REPORT_PATH}[/bold green]")

### Expected Output

```
Audit report written to todo_fixme_report.md
```

* **What learners should see:** Confirmation message indicating successful file save.
* **What indicates success:** `todo_fixme_report.md` exists and contains complete markdown content.


### 💡 Interactive Knowledge Check 2

> [!TIP]
> **Question 2:** Why is `query()` asynchronous (`async for message in query(...)`)?
>
> *Options:*
> A. To allow parallel thread execution of custom Python tools.  
> B. Because network API calls and multi-turn local tool execution are streaming I/O operations, allowing real-time event processing without blocking the host thread.  
> C. It is a standard requirement for all Anthropic Python APIs regardless of function.  
>
> <details>
> <summary><b>Click to reveal Answer & Explanation</b></summary>
> <b>Correct Answer: B</b><br/>
> Agent loops involve asynchronous streaming network requests and non-blocking tool execution. Using an async generator interface allows calling applications to monitor progress events live without thread blocking.
> </details>


# 7. Educational Deep Dives: SDK Runtime & Mechanics

---

### Deep Dive 7.1: The `query()` Asynchronous Interface

`query()` is the primary execution interface of the Anthropic Agent SDK:

```python
async for message in query(
    prompt="Scan src/ for TODO annotations",
    options=ClaudeAgentOptions(allowed_tools=["Read", "Glob", "Grep"])
):
    if isinstance(message, ResultMessage):
        print(message.result)
```

It returns an asynchronous generator yielding structured message events as execution unfolds:
- `SystemMessage`: Emitted on session initialization (subtype `"init"`).
- `UserMessage`: Represents user task inputs.
- `AssistantMessage`: Emitted on model turns; contains text or `tool_use` requests.
- `ResultMessage`: Emitted on loop completion; contains status subtype (`"success"`, `"error_max_turns"`, etc.) and final `.result`.

---

### Deep Dive 7.2: `tool_use` and `tool_result` Protocol

When Claude requests a tool call, the SDK executes the following protocol under the hood:

1. **Emission:** Claude generates a `tool_use` block with tool name and input arguments:
   ```json
   {
     "type": "tool_use",
     "id": "toolu_01A2B3C4",
     "name": "Grep",
     "input": {"pattern": "TODO|FIXME", "path": "data/**/*.py"}
   }
   ```
2. **Permission Verification:** The SDK verifies `"Grep"` against `options.allowed_tools`.
3. **Local Dispatch:** The SDK executes the corresponding local built-in tool function.
4. **Observation Serialization:** The result is wrapped into a `tool_result` block:
   ```json
   {
     "type": "tool_result",
     "tool_use_id": "toolu_01A2B3C4",
     "content": "data/app.py:6: # TODO: Add SSL..."
   }
   ```
5. **Context Ingestion:** The SDK appends `tool_result` to history and triggers the next reasoning turn.

---

### Deep Dive 7.3: Automatic State Management & Turn Lifecycle

In traditional API development, developers maintain a `messages` list manually. In the Agent SDK:
* **Context Accumulation:** The SDK maintains conversation context across turns automatically.
* **Auto-Compaction:** As conversation history grows toward context window limits, the SDK auto-compacts prior tool outputs while preserving reasoning context.
* **Turn Definition:** One turn = model reasoning → tool request → local execution → tool-result ingestion. Read-only tools requested in a single turn execute concurrently.

---

### Deep Dive 7.4: Permission Boundaries (`ClaudeAgentOptions`)

`ClaudeAgentOptions` establishes security boundaries:

```python
options = ClaudeAgentOptions(
    allowed_tools=["Read", "Glob", "Grep"], # Explicit tool whitelist
    model="claude-sonnet-4-5",               # Pinned model selection
    max_turns=15,                           # Turn count safety limit
    max_budget_usd=1.00                     # Cost safety cap
)
```

Restricting capabilities via `allowed_tools` provides **defense-in-depth**. Any tool requested by Claude that is omitted from `allowed_tools` is denied execution automatically by the SDK runtime.


# 8. Complete Walkthrough: Production Local Explorer Agent

Below is the complete, self-contained implementation combining option initialization, task definition, streaming loop execution, error handling, and rich formatting into a single script.


In [ ]:
import os
import asyncio
from dotenv import load_dotenv
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, ResultMessage
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown

async def run_production_local_explorer(target_path: str, report_filename: str):
    console = Console()
    console.print(Panel("[bold white]Anthropic Agent SDK — Production Local Explorer[/bold white]", expand=False))

    # 1. Verify API Authorization Key
    load_dotenv()
    if not os.getenv("ANTHROPIC_API_KEY"):
        console.print("[bold red]Error: ANTHROPIC_API_KEY environment variable is not set.[/bold red]")
        return ""

    # 2. Configure Execution Options & Security Boundaries
    options = ClaudeAgentOptions(
        allowed_tools=["Read", "Glob", "Grep"],
        model="claude-sonnet-4-5"
    )

    # 3. Define Natural Language Task
    task_prompt = f"""
    Perform a complete codebase health audit on the directory '{target_path}'.
    Search for all TODO and FIXME comments.
    Format your response as a structured markdown report with file paths, line numbers, and issue summaries.
    """

    console.print(f"[cyan]Target Directory:[/cyan] {target_path}")
    console.print(f"[cyan]Allowed Tools Whitelist:[/cyan] {options.allowed_tools}")
    console.print("[bold yellow]Executing agent query loop...[/bold yellow]")

    final_report = ""
    turns_executed = 0

    # 4. Stream Loop Execution via query()
    async for message in query(prompt=task_prompt, options=options):
        if isinstance(message, AssistantMessage):
            tool_calls = [
                block.name for block in message.content 
                if hasattr(block, "type") and block.type == "tool_use"
            ]
            if tool_calls:
                turns_executed += 1
                console.print(f"  [dim]Turn {turns_executed}: Executing tool(s) -> {', '.join(tool_calls)}[/dim]")

        elif isinstance(message, ResultMessage):
            if message.subtype == "success":
                final_report = message.result or ""
                console.print(f"[bold green]✓ Agent completed task in {turns_executed} tool turn(s).[/bold green]")
            else:
                console.print(f"[bold red]✗ Agent execution terminated with status: {message.subtype}[/bold red]")
                return ""

    # 5. Render & Save Report
    console.print(Panel(Markdown(final_report), title="[bold green]Audit Report Summary[/bold green]"))

    with open(report_filename, "w", encoding="utf-8") as f:
        f.write(final_report)

    console.print(f"[bold cyan]Report saved to disk: '{report_filename}'[/bold cyan]")
    return final_report


# Execute complete walkthrough with graceful failure handling
try:
    response_text = await run_production_local_explorer("data", "final_code_audit.md")
except Exception as e:
    response_text = f"Execution failed: {e}"
    console.print(f"[bold red]✗ Execution failed: {e}[/bold red]")

console.print("\n[bold cyan]--- Agent Response ---[/bold cyan]\n")
console.print(Markdown(response_text or "No report was generated."))

### Expected Output

```
┌──────────────────────────────────────────────────────────┐
│ Anthropic Agent SDK — Production Local Explorer          │
└──────────────────────────────────────────────────────────┘
Target Directory: data
Allowed Tools Whitelist: ['Read', 'Glob', 'Grep']

Executing agent query loop...
  Turn 1: Executing tool(s) -> Glob
  Turn 2: Executing tool(s) -> Grep
  Turn 3: Executing tool(s) -> Read

✓ Agent completed task in 3 tool turn(s).

[Rendered Markdown Report Panel]
Report saved to disk: 'final_code_audit.md'
```


# 9. Experimentation: Try It Yourself

Extend the agent workflow by modifying parameters, tool whitelists, and model settings.

---

### Experiment 1: Expand Search Targets

* **Goal:** Extend search targets to locate `HACK`, `DEPRECATED`, or `XXX` annotations.
* **Modification:** Update task prompt:
  ```python
  TASK = "Scan 'data' and locate all TODO, FIXME, HACK, DEPRECATED, and XXX annotations."
  ```
* **Expected Behavior:** Claude expands `Grep` regex patterns to search for all five keywords across files.
* **What to Observe:** How Claude adjusts its grep regex expression automatically without requiring changes to tool code.
* **Suggested Discussion:** How does autonomous pattern search reduce tool development overhead?

---

### Experiment 2: Permission Testing Boundaries

* **Goal:** Add `Edit` or `Write` to `allowed_tools` and observe agent planning.
* **Modification:** Add `"Edit"` to `allowed_tools` and prompt Claude to fix a `TODO` item:
  ```python
  options = ClaudeAgentOptions(allowed_tools=["Read", "Glob", "Grep", "Edit"])
  ```
* **Expected Behavior:** Claude will locate the TODO item, inspect surrounding context, and emit an `Edit` tool call to modify the file.
* **What to Observe:** If `Edit` is omitted from `allowed_tools`, Claude explicitly states it cannot modify files due to tool restrictions.
* **Suggested Discussion:** Why are strict permission boundaries critical for production enterprise AI workflows?

---

### Experiment 3: Turn & Budget Limits

* **Goal:** Test early termination behavior and handle `error_max_turns`.
* **Modification:** Set `max_turns=2` or `max_budget_usd=0.05` on `ClaudeAgentOptions`:
  ```python
  options = ClaudeAgentOptions(allowed_tools=["Read", "Glob", "Grep"], max_turns=2)
  ```
* **Expected Behavior:** Execution stops early after 2 turns, yielding `ResultMessage` with `subtype="error_max_turns"`.
* **What to Observe:** How application error logic handles non-success subtypes safely.
* **Suggested Discussion:** How do turn limits and cost budgets safeguard enterprise applications against runaway loops?

---

### Experiment 4: Reasoning Depth Control

* **Goal:** Adjust reasoning effort (`effort="low"`, `"medium"`, or `"high"`) on options.
* **Modification:** Set `effort` in `ClaudeAgentOptions`.
* **Expected Behavior:** High effort increases pre-action reasoning quality; low effort reduces execution latency.
* **Suggested Discussion:** When should high reasoning effort be enabled in autonomous coding agents?

---

### Experiment 5: Model Selection

* **Goal:** Compare response quality and speed across Claude models.
* **Modification:** Swap `model="claude-sonnet-4-5"` for `model="claude-sonnet-5"` in `ClaudeAgentOptions`.
* **Expected Behavior:** Compare report formatting detail, turn efficiency, and execution speed across model versions. See the [Anthropic models reference](https://platform.claude.com/docs/en/about-claude/models) for available model identifiers.


# 10. Key Takeaways

---

### Summary Matrix

| Concept | Key Takeaway |
|---------|--------------|
| **Agent SDK vs. Client SDK** | Client SDK requires manual loop orchestration and state management; Agent SDK automates the entire multi-turn tool cycle via `query()`. |
| **The Agent Loop** | Iterative cycle: model evaluation → tool-call emission → local tool execution → observation ingestion. |
| **`query()` Interface** | Single async generator call managing prompt evaluation, local tool execution, state updates, and termination conditions. |
| **Built-in Tools** | Native tools like `Read`, `Glob`, and `Grep` give agents secure filesystem exploration capabilities out of the box. |
| **`allowed_tools` Boundary** | Explicit tool permissioning defines strict operational boundaries, enabling safe autonomous execution. |
| **Turn Lifecycle** | Each cycle of model evaluation, tool execution, and result ingestion forms a turn, continuing until Claude returns a final text response. |
| **`ResultMessage` Inspection** | Always verify `message.subtype == "success"` before accessing `.result` to handle early terminations safely. |


# 11. Further Reading & Official Resources

Consult official Anthropic documentation for additional technical details:

* **Anthropic Console:** [console.anthropic.com](https://console.anthropic.com) — Manage API keys, organization settings, and monitor usage metrics.
* **Anthropic Models Reference:** [platform.claude.com/docs/en/about-claude/models](https://platform.claude.com/docs/en/about-claude/models) — Complete reference for available Claude model IDs, specifications, and capabilities.
